# GroupDNA - WhatsApp Group Chat Analyzer

**Project Name:** GroupDNA - Your WhatsApp Group, Decoded

**Student Name:** [Your Name]

**Roll Number:** DS24007

**Batch:** Data Science June 2024

**Date:** June 27, 2026

---

## Project Overview
This project analyzes a WhatsApp group chat export to generate a comprehensive personality and activity report. It identifies patterns in messaging behavior, assigns personality archetypes to participants, and visualizes group dynamics through text-based analytics.

**Dataset:** hostel_bois.txt (synthetic WhatsApp chat export)

**Participants:** 6 (Rahul, Priya, Aman, Karan, Neha, Vikas)

**Date Range:** 01 April 2024 to 30 May 2024 (60 days)

**Total Messages:** 3,174

## Feature 1: Chat Parser

This feature reads the WhatsApp chat export file line by line, extracts timestamp, sender, and message text, and handles edge cases (system messages, media-omitted, deleted messages).

In [ ]:
# Feature 1: Chat Parser
# AI-assisted: Used for understanding datetime parsing and file structure

from datetime import datetime
import numpy as np

def parse_chat(file_path):
    """
    Parse WhatsApp chat export file and extract messages.
    
    Args:
        file_path: Path to the WhatsApp chat export file
    
    Returns:
        messages: List of dictionaries with keys 'timestamp', 'sender', 'message'
        stats: Dictionary with parsing statistics
    """
    messages = []
    stats = {
        'total_lines': 0,
        'real_messages': 0,
        'system_messages': 0,
        'media_omitted': 0,
        'deleted_messages': 0
    }
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    lines = content.split('\n')
    stats['total_lines'] = len(lines)
    
    for line in lines:
        # Skip empty lines
        if not line.strip():
            continue
        
        # Check if line starts with date pattern (DD/MM/YY)
        if len(line) < 8 or line[2] != '/' or line[5] != '/':
            # This is a continuation line (multi-line message)
            # In our dataset, all messages are single-line, so we skip
            continue
        
        # Split timestamp from rest of line
        try:
            timestamp_part, rest = line.split(' - ', 1)
        except ValueError:
            continue
        
        # Check for system messages (no colon after timestamp)
        if ':' not in rest:
            stats['system_messages'] += 1
            continue
        
        # Split sender from message
        try:
            sender, message = rest.split(': ', 1)
        except ValueError:
            continue
        
        # Check for media-omitted
        if '<Media omitted>' in message:
            stats['media_omitted'] += 1
            continue
        
        # Check for deleted messages
        if 'This message was deleted' in message:
            stats['deleted_messages'] += 1
            continue
        
        # Valid message - add to list
        messages.append({
            'timestamp': timestamp_part,
            'sender': sender,
            'message': message
        })
        stats['real_messages'] += 1
    
    return messages, stats

# Parse the chat file
messages, parse_stats = parse_chat('hostel_bois.txt')

print(f"Successfully parsed {parse_stats['real_messages']} messages from {len(set(m['sender'] for m in messages))} participants over 60 days")
print(f"Skipped: {parse_stats['system_messages']} system messages, {parse_stats['media_omitted']} media-omitted, {parse_stats['deleted_messages']} deleted messages")

## Feature 2: Group Overview

Computes and displays headline statistics about the group including total messages, date range, participants, and per-person message counts.

In [ ]:
# Feature 2: Group Overview

def compute_group_overview(messages):
    """
    Compute group statistics and overview.
    
    Args:
        messages: List of message dictionaries
    
    Returns:
        Dictionary with group statistics
    """
    if not messages:
        return {}
    
    # Get unique participants
    participants = set(m['sender'] for m in messages)
    
    # Count messages per person
    message_counts = {}
    for msg in messages:
        sender = msg['sender']
        message_counts[sender] = message_counts.get(sender, 0) + 1
    
    # Sort by message count (descending)
    sorted_counts = sorted(message_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Get date range
    timestamps = [datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M') for m in messages]
    first_date = min(timestamps)
    last_date = max(timestamps)
    
    # Calculate total days
    total_days = (last_date - first_date).days + 1
    
    return {
        'group_name': 'Hostel Bois 4ever',
        'first_date': first_date.strftime('%d %B %Y'),
        'last_date': last_date.strftime('%d %B %Y'),
        'total_days': total_days,
        'total_messages': len(messages),
        'participants': len(participants),
        'message_counts': sorted_counts
    }

# Compute overview
overview = compute_group_overview(messages)

# Print formatted overview
print("=" * 60)
print("  GROUP OVERVIEW")
print("=" * 60)
print(f"  Group           : {overview['group_name']}")
print(f"  Period          : {overview['first_date']} to {overview['last_date']} ({overview['total_days']} days)")
print(f"  Total messages  : {overview['total_messages']:,}")
print(f"  Participants    : {overview['participants']}")
print()
print("  MESSAGES PER PERSON")
total_msgs = overview['total_messages']
for sender, count in overview['message_counts']:
    percentage = (count / total_msgs) * 100
    print(f"    {sender:<10} : {count:>4}  ({percentage:>5.1f}%)")

## Feature 3: Word Frequency Analysis

Extracts and counts words from all messages, identifies the top 10 most-used words, and displays them with visual bars.

In [ ]:
# Feature 3: Word Frequency Analysis

def analyze_word_frequency(messages):
    """
    Analyze word frequency across all messages.
    
    Args:
        messages: List of message dictionaries
    
    Returns:
        List of tuples (word, count) for top words
    """
    # Define stop words to skip
    stop_words = {
        'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
        'with', 'at', 'by', 'from', 'as', 'be', 'this', 'that', 'it', 'not',
        'are', 'was', 'were', 'have', 'has', 'had', 'do', 'does', 'did',
        'but', 'if', 'so', 'my', 'your', 'we', 'you', 'he', 'she', 'they',
        'me', 'him', 'her', 'them', 'us', 'what', 'when', 'where', 'who',
        'why', 'how', 'just', 'like', 'get', 'got', 'go', 'going', 'went'
    }
    
    # Punctuation to strip
    punctuation = ".,!?;:'\"()[]{}<>-_/\\|@#$%^&*~`"
    
    word_counts = {}
    
    for msg in messages:
        text = msg['message'].lower()
        
        # Strip punctuation
        for char in punctuation:
            text = text.replace(char, ' ')
        
        # Split into words
        words = text.split()
        
        # Count words (skip stop words and short words)
        for word in words:
            if len(word) < 2 or word in stop_words:
                continue
            word_counts[word] = word_counts.get(word, 0) + 1
    
    # Get top 10 words
    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    
    return sorted_words

# Analyze word frequency
top_words = analyze_word_frequency(messages)

# Print formatted word frequency
print("  THIS GROUP'S FAVOURITE WORDS")
max_count = top_words[0][1] if top_words else 1
for word, count in top_words:
    bar_length = int((count / max_count) * 30)
    bar = '█' * bar_length
    print(f"    {word:<12} {bar:<30} {count}")

## Feature 4: NumPy Activity Heatmap

Creates a text-based heatmap showing message activity by hour for each participant using NumPy.

In [ ]:
# Feature 4: NumPy Activity Heatmap

def create_activity_heatmap(messages):
    """
    Create a NumPy-based activity heatmap showing messages by hour.
    
    Args:
        messages: List of message dictionaries
    
    Returns:
        Tuple of (participants_list, hour_labels, numpy_matrix)
    """
    # Get unique participants (sorted)
    participants = sorted(set(m['sender'] for m in messages))
    
    # Define hour bins (3-hour intervals for cleaner display)
    hour_bins = [0, 3, 6, 9, 12, 15, 18, 21]
    hour_labels = ['00', '03', '06', '09', '12', '15', '18', '21']
    
    # Initialize NumPy matrix
    matrix = np.zeros((len(participants), len(hour_bins)), dtype=int)
    
    # Count messages per person per hour bin
    for msg in messages:
        sender = msg['sender']
        timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        hour = timestamp.hour
        
        # Find which bin this hour belongs to
        bin_idx = 0
        for i, bin_start in enumerate(hour_bins):
            if hour >= bin_start:
                bin_idx = i
            else:
                break
        
        row_idx = participants.index(sender)
        matrix[row_idx, bin_idx] += 1
    
    return participants, hour_labels, matrix

# Create heatmap
participants, hour_labels, activity_matrix = create_activity_heatmap(messages)

# Print formatted heatmap
print("  ACTIVITY HEATMAP (messages by hour)")
print("           ", end="")
for label in hour_labels:
    print(f"  {label} ", end="")
print()

for i, person in enumerate(participants):
    print(f"  {person:<10}", end="")
    row_max = activity_matrix[i].max() if activity_matrix[i].max() > 0 else 1
    for j in range(len(hour_labels)):
        value = activity_matrix[i, j]
        if value == 0:
            char = '.'
        elif value < row_max * 0.25:
            char = '.'
        elif value < row_max * 0.5:
            char = '░'
        elif value < row_max * 0.75:
            char = '▒'
        else:
            char = '█'
        print(f"  {char}  ", end="")
    print()

# Print row totals
print("\n  Row totals (messages per person):")
for i, person in enumerate(participants):
    print(f"    {person:<10} : {activity_matrix[i].sum()}")

## Feature 5: Response Times & Silent Streaks

Computes average response time for each participant and identifies their longest silent streak (consecutive days with zero messages).

## Feature 5.5: Most Active Day and Hour

Identifies the most active day and hour across the entire group chat period.

In [ ]:
# Feature 5.5: Most Active Day and Hour

def find_most_active_periods(messages):
    """
    Find the most active day and hour in the group chat.
    
    Args:
        messages: List of message dictionaries
    
    Returns:
        Dictionary with most active day and hour information
    """
    if not messages:
        return {}
    
    # Count messages per day
    day_counts = {}
    hour_counts = {}
    
    # Get all unique dates to calculate total days
    unique_dates = set()
    
    for msg in messages:
        timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        date = timestamp.date()
        hour = timestamp.hour
        
        unique_dates.add(date)
        
        # Count per day
        day_str = date.strftime('%d %B %Y')
        day_counts[day_str] = day_counts.get(day_str, 0) + 1
        
        # Count per hour
        hour_counts[hour] = hour_counts.get(hour, 0) + 1
    
    # Find most active day
    most_active_day = max(day_counts.items(), key=lambda x: x[1])
    
    # Find most active hour
    most_active_hour = max(hour_counts.items(), key=lambda x: x[1])
    
    # Calculate total days for average calculation
    total_days = len(unique_dates)
    
    # Calculate average messages per day for the busiest hour
    avg_messages_per_day = most_active_hour[1] / total_days if total_days > 0 else 0
    
    return {
        'most_active_day': {
            'date': most_active_day[0],
            'count': most_active_hour[1]
        },
        'most_active_hour': {
            'hour': most_active_hour[0],
            'count': most_active_hour[1],
            'avg_per_day': avg_messages_per_day
        },
        'day_counts': day_counts,
        'hour_counts': hour_counts
    }

# Find most active periods
activity_periods = find_most_active_periods(messages)

# Print formatted output matching the requested format
print(f"Busiest day  : {activity_periods['most_active_day']['date']} ({activity_periods['most_active_day']['count']} messages)")
hour = activity_periods['most_active_hour']['hour']
next_hour = (hour + 1) % 24
avg = activity_periods['most_active_hour']['avg_per_day']
print(f"Busiest hour : {hour:02d}:00 - {next_hour:02d}:00 (avg {avg:.1f} messages per day)")

In [ ]:
# Feature 5: Response Times & Silent Streaks

def compute_response_patterns(messages):
    """
    Compute response times and silent streaks for each participant.
    
    Args:
        messages: List of message dictionaries
    
    Returns:
        Dictionary with response times and silent streaks
    """
    if not messages:
        return {}
    
    # Sort messages by timestamp
    sorted_messages = sorted(messages, key=lambda x: datetime.strptime(x['timestamp'], '%d/%m/%y, %H:%M'))
    
    # Get participants
    participants = set(m['sender'] for m in messages)
    
    # Initialize data structures
    response_times = {p: [] for p in participants}
    active_days = {p: set() for p in participants}
    
    # Compute response times
    for i in range(1, len(sorted_messages)):
        current_msg = sorted_messages[i]
        prev_msg = sorted_messages[i-1]
        
        # Only compute if different sender
        if current_msg['sender'] != prev_msg['sender']:
            current_time = datetime.strptime(current_msg['timestamp'], '%d/%m/%y, %H:%M')
            prev_time = datetime.strptime(prev_msg['timestamp'], '%d/%m/%y, %H:%M')
            gap_seconds = (current_time - prev_time).total_seconds()
            response_times[current_msg['sender']].append(gap_seconds)
    
    # Track active days for each person
    for msg in sorted_messages:
        timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        date = timestamp.date()
        active_days[msg['sender']].add(date)
    
    # Compute average response times
    avg_response_times = {}
    for person in participants:
        times = response_times[person]
        if times:
            avg_response_times[person] = sum(times) / len(times)
        else:
            avg_response_times[person] = float('inf')
    
    # Compute longest silent streaks
    all_dates = sorted(set(m.date() for m in [datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M') for msg in messages]))
    silent_streaks = {}
    
    for person in participants:
        person_active = active_days[person]
        max_streak = 0
        current_streak = 0
        streak_start = None
        streak_end = None
        
        for date in all_dates:
            if date not in person_active:
                current_streak += 1
                if streak_start is None:
                    streak_start = date
                streak_end = date
            else:
                if current_streak > max_streak:
                    max_streak = current_streak
                current_streak = 0
                streak_start = None
        
        # Check final streak
        if current_streak > max_streak:
            max_streak = current_streak
        
        silent_streaks[person] = {
            'days': max_streak,
            'start': streak_start if max_streak > 0 else None,
            'end': streak_end if max_streak > 0 else None
        }
    
    return {
        'avg_response_times': avg_response_times,
        'silent_streaks': silent_streaks
    }

# Compute response patterns
response_data = compute_response_patterns(messages)

# Find fastest and slowest repliers
avg_times = response_data['avg_response_times']
fastest = min(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else float('inf'))
slowest = max(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else 0)

# Print response patterns
print("  RESPONSE PATTERNS")
if fastest[1] != float('inf'):
    fastest_minutes = fastest[1] / 60
    print(f"    Fastest replier : {fastest[0]} (avg {fastest_minutes:.1f} minutes)")
else:
    print(f"    Fastest replier : N/A")

if slowest[1] != float('inf'):
    if slowest[1] >= 3600:
        slowest_hours = slowest[1] / 3600
        print(f"    Slowest replier : {slowest[0]} (avg {slowest_hours:.1f} hours)")
    else:
        slowest_minutes = slowest[1] / 60
        print(f"    Slowest replier : {slowest[0]} (avg {slowest_minutes:.1f} minutes)")
else:
    print(f"    Slowest replier : N/A")

print()
print("  LONGEST SILENT STREAKS (consecutive days with zero messages)")

# Sort by silent streak length
silent_streaks = response_data['silent_streaks']
sorted_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['days'], reverse=True)

for person, streak in sorted_streaks:
    if streak['days'] == 0:
        print(f"    {person:<10} :  0 days  (never went silent)")
    elif streak['days'] == 1:
        print(f"    {person:<10} :  {streak['days']} day")
    else:
        if streak['start'] and streak['end']:
            start_str = streak['start'].strftime('%d %b')
            end_str = streak['end'].strftime('%d %b')
            print(f"    {person:<10} : {streak['days']:>2} days ({start_str} to {end_str})")
        else:
            print(f"    {person:<10} : {streak['days']:>2} days")

## Feature 6: Personality Archetype Detection

Assigns personality archetypes to each participant based on quantitative rules including: Spammer, Group Mom, Night Owl, Storyteller, Drama Queen, Ghost, Comedian, and Question Master.

In [ ]:
# Feature 6: Personality Archetype Detection

def detect_archetypes(messages):
    """
    Detect personality archetypes for each participant.
    
    Args:
        messages: List of message dictionaries
    
    Returns:
        Dictionary mapping each person to their archetype and score
    """
    participants = set(m['sender'] for m in messages)
    
    # Sort messages by timestamp for analysis
    sorted_messages = sorted(messages, key=lambda x: datetime.strptime(x['timestamp'], '%d/%m/%y, %H:%M'))
    
    # Initialize scores for each archetype
    archetype_scores = {p: {} for p in participants}
    
    # --- THE SPAMMER: Avg consecutive message burst > 3 ---
    for person in participants:
        person_messages = [m for m in sorted_messages if m['sender'] == person]
        bursts = []
        current_burst = 1
        
        for i in range(1, len(person_messages)):
            # Find position in overall message list
            current_idx = sorted_messages.index(person_messages[i])
            prev_idx = sorted_messages.index(person_messages[i-1])
            
            # If consecutive in overall list (no one else spoke)
            if current_idx == prev_idx + 1:
                current_burst += 1
            else:
                bursts.append(current_burst)
                current_burst = 1
        
        bursts.append(current_burst)
        avg_burst = sum(bursts) / len(bursts) if bursts else 0
        archetype_scores[person]['SPAMMER'] = avg_burst
    
    # --- THE GROUP MOM: Caring keywords ---
    caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 
                      'reminder', 'drink water', 'don\'t forget', 'careful', 'well']
    
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        caring_count = 0
        
        for msg in person_messages:
            text_lower = msg['message'].lower()
            for keyword in caring_keywords:
                if keyword in text_lower:
                    caring_count += 1
                    break
        
        percentage = (caring_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['GROUP_MOM'] = percentage
    
    # --- THE NIGHT OWL: Messages between 23:00 and 04:59 ---
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        night_count = 0
        
        for msg in person_messages:
            timestamp = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
            hour = timestamp.hour
            if hour >= 23 or hour < 5:
                night_count += 1
        
        percentage = (night_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['NIGHT_OWL'] = percentage
    
    # --- THE STORYTELLER: Average words per message > 30 ---
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        total_words = 0
        
        for msg in person_messages:
            words = msg['message'].split()
            total_words += len(words)
        
        avg_words = total_words / len(person_messages) if person_messages else 0
        archetype_scores[person]['STORYTELLER'] = avg_words
    
    # --- THE DRAMA QUEEN: ALL-CAPS or multiple exclamations ---
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        drama_count = 0
        
        for msg in person_messages:
            text = msg['message']
            # Check if all caps (and length > 3)
            if len(text) > 3 and text.isupper() and text.isalpha():
                drama_count += 1
            # Check for 2+ exclamations
            elif text.count('!') >= 2:
                drama_count += 1
        
        percentage = (drama_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['DRAMA_QUEEN'] = percentage
    
    # --- THE GHOST: Silent on > 60% of days ---
    all_dates = sorted(set(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M').date() for m in messages))
    total_days = len(all_dates)
    
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        active_dates = set(datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M').date() for m in person_messages)
        silent_days = total_days - len(active_dates)
        silent_percentage = (silent_days / total_days) * 100 if total_days > 0 else 0
        archetype_scores[person]['GHOST'] = silent_percentage
    
    # --- THE COMEDIAN: Laughter words ---
    laughter_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        laughter_count = 0
        
        for msg in person_messages:
            text_lower = msg['message'].lower()
            for word in laughter_words:
                if word in text_lower:
                    laughter_count += 1
                    break
        
        percentage = (laughter_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['COMEDIAN'] = percentage
    
    # --- THE QUESTION MASTER: Messages ending with '?' ---
    for person in participants:
        person_messages = [m for m in messages if m['sender'] == person]
        question_count = 0
        
        for msg in person_messages:
            if msg['message'].strip().endswith('?'):
                question_count += 1
        
        percentage = (question_count / len(person_messages)) * 100 if person_messages else 0
        archetype_scores[person]['QUESTION_MASTER'] = percentage
    
    # Assign best archetype to each person
    archetype_names = {
        'SPAMMER': 'THE SPAMMER',
        'GROUP_MOM': 'THE GROUP MOM',
        'NIGHT_OWL': 'THE NIGHT OWL',
        'STORYTELLER': 'THE STORYTELLER',
        'DRAMA_QUEEN': 'THE DRAMA QUEEN',
        'GHOST': 'THE GHOST',
        'COMEDIAN': 'THE COMEDIAN',
        'QUESTION_MASTER': 'THE QUESTION MASTER'
    }
    
    assignments = {}
    for person in participants:
        scores = archetype_scores[person]
        best_archetype = max(scores.items(), key=lambda x: x[1])
        assignments[person] = {
            'archetype': archetype_names[best_archetype[0]],
            'score': best_archetype[1]
        }
    
    return assignments, archetype_scores

# Detect archetypes
archetype_assignments, all_scores = detect_archetypes(messages)

# Print archetype assignments
print("  PERSONALITY ARCHETYPES")
for person in sorted(archetype_assignments.keys()):
    assignment = archetype_assignments[person]
    archetype = assignment['archetype']
    score = assignment['score']
    
    # Format score based on archetype
    if 'SPAMMER' in archetype:
        print(f"    {person:<10} →  {archetype:<20} (avg {score:.1f} msgs in a row)")
    elif 'GROUP MOM' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% caring keywords)")
    elif 'NIGHT OWL' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% msgs after 11 PM)")
    elif 'STORYTELLER' in archetype:
        print(f"    {person:<10} →  {archetype:<20} (avg {score:.1f} words per msg)")
    elif 'DRAMA QUEEN' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% ALL-CAPS msgs)")
    elif 'GHOST' in archetype:
        print(f"    {person:<10} →  {archetype:<20} (silent {score:.1f}% of days)")
    elif 'COMEDIAN' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% laughter)")
    elif 'QUESTION MASTER' in archetype:
        print(f"    {person:<10} →  {archetype:<20} ({score:.1f}% questions)")

## Feature 7: Final Report

Generates a comprehensive, beautifully formatted report combining all features with proper visual hierarchy and formatting.

In [ ]:
# Feature 7: Final Report

def generate_final_report(messages, overview, top_words, participants, hour_labels, activity_matrix, 
                          response_data, archetype_assignments, activity_periods):
    """
    Generate the final formatted report matching the exact format from the image.
    """
    print("=" * 70)
    print(f'GROUPDNA REPORT - "{overview["group_name"]}"')
    print(f"{overview['total_days']} days • {overview['total_messages']:,} messages • {overview['participants']} members")
    print("=" * 70)
    print()
    
    # Period and Busiest Times Section
    print(f"Period         : {overview['first_date']} to {overview['last_date']}")
    print(f"Busiest day    : {activity_periods['most_active_day']['date']} ({activity_periods['most_active_day']['count']} messages)")
    hour = activity_periods['most_active_hour']['hour']
    next_hour = (hour + 1) % 24
    avg = activity_periods['most_active_hour']['avg_per_day']
    print(f"Busiest hour   : {hour:02d}:00 - {next_hour:02d}:00 (avg {avg:.1f} messages per day)")
    print()
    
    # Messages Per Person Section
    print("MESSAGES PER PERSON")
    total_msgs = overview['total_messages']
    max_bar_length = 20
    for sender, count in overview['message_counts']:
        percentage = (count / total_msgs) * 100
        bar_length = int((percentage / 100) * max_bar_length)
        bar = '█' * bar_length
        print(f"{sender:<10} {bar:<{max_bar_length}} {count:>5} ({percentage:>5.1f}%)")
    print()
    
    # Activity Heatmap Section
    print("ACTIVITY HEATMAP")
    print("           ", end="")
    for label in hour_labels:
        print(f"  {label} ", end="")
    print()
    
    for i, person in enumerate(participants):
        print(f"  {person:<10}", end="")
        row_max = activity_matrix[i].max() if activity_matrix[i].max() > 0 else 1
        for j in range(len(hour_labels)):
            value = activity_matrix[i, j]
            if value == 0:
                char = '.'
            elif value < row_max * 0.25:
                char = '.'
            elif value < row_max * 0.5:
                char = '░'
            elif value < row_max * 0.75:
                char = '▒'
            else:
                char = '█'
            print(f"  {char}  ", end="")
        
        # Add archetype annotation for Night Owl
        assignment = archetype_assignments[person]
        if 'NIGHT OWL' in assignment['archetype']:
            print(" <- NIGHT OWL", end="")
        print()
    print()
    
    # Top Words Section (Top 5 instead of Top 10)
    print("THIS GROUP'S FAVOURITE WORDS")
    top_5_words = top_words[:5]
    max_count = top_5_words[0][1] if top_5_words else 1
    for word, count in top_5_words:
        bar_length = int((count / max_count) * 20)
        bar = '█' * bar_length
        print(f"{word:<12} {bar:<20} {count}")
    print()
    
    # Response Patterns Section
    print("RESPONSE PATTERNS")
    avg_times = response_data['avg_response_times']
    fastest = min(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else float('inf'))
    slowest = max(avg_times.items(), key=lambda x: x[1] if x[1] != float('inf') else 0)
    
    if fastest[1] != float('inf'):
        fastest_minutes = fastest[1] / 60
        print(f"Fastest replier : {fastest[0]} (avg {fastest_minutes:.1f} minutes)")
    else:
        print(f"Fastest replier : N/A")
    
    if slowest[1] != float('inf'):
        if slowest[1] >= 3600:
            slowest_hours = slowest[1] / 3600
            print(f"Slowest replier : {slowest[0]} (avg {slowest_hours:.1f} hours)")
        else:
            slowest_minutes = slowest[1] / 60
            print(f"Slowest replier : {slowest[0]} (avg {slowest_minutes:.1f} minutes)")
    else:
        print(f"Slowest replier : N/A")
    print()
    
    # Longest Silent Streaks Section
    print("LONGEST SILENT STREAKS")
    silent_streaks = response_data['silent_streaks']
    sorted_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['days'], reverse=True)
    
    for person, streak in sorted_streaks:
        if streak['days'] == 0:
            continue  # Skip people who never went silent
        elif streak['days'] == 1:
            print(f"{person:<10} : {streak['days']} day")
        else:
            if streak['start'] and streak['end']:
                start_str = streak['start'].strftime('%d %b')
                end_str = streak['end'].strftime('%d %b')
                print(f"{person:<10} : {streak['days']} days ({start_str} - {end_str})")
            else:
                print(f"{person:<10} : {streak['days']} days")
    print()

# Generate final report
generate_final_report(messages, overview, top_words, participants, hour_labels, 
                      activity_matrix, response_data, archetype_assignments, activity_periods)

---

## Reflection

### What was the hardest part?
The most challenging part of this project was implementing the archetype detection logic, particularly the consecutive message burst calculation for the Spammer archetype. Getting the burst detection to correctly identify when someone sends multiple messages in a row without others speaking required careful index tracking and edge case handling.

### What would you do differently?
If I were to redo this project, I would:
1. Start with unit tests for each feature to catch bugs earlier
2. Create helper functions for common operations like datetime parsing to reduce code duplication
3. Add more robust error handling for edge cases in the chat format
4. Implement the multi-line message handling from the start (even though the dataset doesn't need it)

### What archetype did I get?
[Optional: Run this on your own chat and share your archetype here!]

---

## Project Constraints Followed

- ✅ No pandas used for file reading (used Python's built-in `open()`)
- ✅ No matplotlib used for heatmap (used text-based rendering with NumPy)
- ✅ No collections.Counter used (built custom word counter with dict)
- ✅ Only Python fundamentals + NumPy + datetime used
- ✅ All 8 features implemented
- ✅ Proper file I/O with UTF-8 encoding
- ✅ Clean, formatted output with visual hierarchy
- ✅ Proper error handling for edge cases

---

**Note:** This notebook can be run end-to-end by uploading `hostel_bois.txt` to the same directory in Google Colab or placing it next to this .ipynb file locally.